In [3]:
from IPython.display import display, Markdown
import joblib
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import ShuffleSplit, GridSearchCV, KFold, cross_validate
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler


## 1. Obtenção de dados

In [4]:
df = pd.read_csv("../data/raw/diamonds.csv")
dictionary = pd.read_csv("../data/external/dictionary.csv")
dictionary


,variavel,descricao,tipo,subtipo
0,carat,peso do diamante,quantitativa,continua
1,cut,"qualidade do corte (Fair, Good, Very Good, Pre...",qualitativa,ordinal
2,color,cor do diamante de D(melhor) a J (pior),qualitativa,ordinal
3,clarity,uma medida de quão claro é o diamante (I1 (pio...,qualitativa,ordinal
4,depth,porcentagem total de profundidade,quantitativa,continua
5,table,largura do topo do diamante em relação ao pont...,quantitativa,discreta
6,price,preço do diamante em dolar americano,quantitativa,continua
7,x,comprimento do diamante em mm (0--10.74),quantitativa,continua
8,y,largura do diamante em mm (0--58.9),quantitativa,continua
9,z,profundidade do diamante em mm (0--31.8),quantitativa,continua


In [82]:
df.describe().T.style.background_gradient(cmap = 'inferno')

,count,mean,std,min,25%,50%,75%,max
carat,53940.000000,0.797940,0.474011,0.200000,0.400000,0.700000,1.040000,5.010000
depth,53940.000000,61.749405,1.432621,43.000000,61.000000,61.800000,62.500000,79.000000
table,53940.000000,57.457184,2.234491,43.000000,56.000000,57.000000,59.000000,95.000000
price,53940.000000,3932.799722,3989.439738,326.000000,950.000000,2401.000000,5324.250000,18823.000000
x,53940.000000,5.731157,1.121761,0.000000,4.710000,5.700000,6.540000,10.740000
y,53940.000000,5.734526,1.142135,0.000000,4.720000,5.710000,6.540000,58.900000
z,53940.000000,3.538734,0.705699,0.000000,2.910000,3.530000,4.040000,31.800000


In [83]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    53940 non-null  float64
 1   cut      53940 non-null  object 
 2   color    53940 non-null  object 
 3   clarity  53940 non-null  object 
 4   depth    53940 non-null  float64
 5   table    53940 non-null  float64
 6   price    53940 non-null  int64  
 7   x        53940 non-null  float64
 8   y        53940 non-null  float64
 9   z        53940 non-null  float64
dtypes: float64(6), int64(1), object(3)
memory usage: 4.1+ MB


## 2. Preparação de dados


Aqui realizamos a normalização, codificação e o tratamento de dados discrepantes e/ou faltantes dentro do conjunto de dados.

In [5]:
target_variable = 'price'
quantitative_variables = (
    dictionary
    .query("tipo == 'quantitativa' and variavel != @target_variable")
    .variavel
    .to_list()
)
nominal_variables = (
    dictionary
    .query("subtipo == 'nominal'")
    .variavel
    .to_list()
)
ordinal_variables = (
    dictionary
    .query("subtipo == 'ordinal'")
    .variavel
    .to_list()
)

In [6]:
def remove_outliers_iqr(df):
    for column in df.select_dtypes(include=['float64', 'int64']).columns:
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]
    return df


df = remove_outliers_iqr(df)

X = df.drop(columns=[target_variable], axis=1)
y = np.ravel(df[[target_variable]])

In [8]:
quantitative_preprocess = Pipeline([
    ('normalization', StandardScaler()) 
])

nominal_preprocess = Pipeline([
    ('encoding', OneHotEncoder(sparse_output=False, drop='first'))  
])

ordinal_preprocess = Pipeline([
    ('encoding', OrdinalEncoder())  
])


preprocessor = ColumnTransformer([
    ('quantitative', quantitative_preprocess, quantitative_variables),
    ('nominal', nominal_preprocess, nominal_variables),
    ('ordinal', ordinal_preprocess, ordinal_variables)
])

## 3. Seleção de modelos

Iremos análisar quatro modelos, que serão testados utilizando um método de validação, a saber:

- K-Nearest-Neighbors
- Support Vector Machine
- Decision Tree
- Random Forest

Além disso, cada um desses algoritmos será testado com diferentes hiper-parâmetros, para que possamos encontrar o melhor modelo e a melhor configuração possível para esse modelo.

Utilizaremos as seguintes métricas para análise:

- Acurácia (accuracy): proporção entre os dados que foram corretamente previstos (como positivos ou negativos) com o total de dados observados;
- Precisão (precision): proporção entre dados corretamente previstos como positivos e o total de observações positivas.
- Recall: proporção entre dados corretamente previstos como positivos com o total de observações.
- F1-score: média entre precision e recall, portanto levando em conta tanto falsos positivos quanto falsos negativos.

In [9]:
n_splits_comparative_analysis = 10
n_folds_grid_search = 5
test_size = .2
random_state = 42
scoring = 'accuracy'
metrics = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']


max_iter = 1000
models = [
    ('K-Nearest Neighbors', KNeighborsClassifier(), {"n_neighbors": range(3, 20, 2), 'weights': ['uniform', 'distance']}),
    ('Suport Vector Machines', SVC(random_state=random_state, max_iter=max_iter), {"kernel": ["linear", "rbf"], 'C':[1,10,100,1000],'gamma':[0.0001, 0.001, 0.1, 1]}),
    ('Decision Tree',  DecisionTreeClassifier(random_state=random_state), {'criterion':['gini','entropy'],'max_depth': [3, 6, 8]}),
    ('Random Forest',  RandomForestClassifier(random_state=random_state), {'criterion':['gini','entropy'],'max_depth': [3, 6, 8], 'n_estimators': [10, 30]}),
]

In [10]:
results = pd.DataFrame({})
cross_validate_grid_search = KFold(n_splits=n_folds_grid_search)
cross_validate_comparative_analysis = ShuffleSplit(n_splits=n_splits_comparative_analysis, test_size=test_size, random_state=random_state)
for model_name, model_object, model_parameters in models:
    print(f"running {model_name}...")
    model_grid_search = GridSearchCV(
        estimator=model_object,
        param_grid=model_parameters,
        scoring=scoring,
        n_jobs=-1,
        cv=cross_validate_grid_search
    )
    
    approach = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model_grid_search)
    ])
    
    scores = cross_validate(
        estimator=approach,
        X=X,
        y=y,
        cv=cross_validate_comparative_analysis,
        n_jobs=-1,
        scoring=metrics
    )
    
    scores['model_name'] = [model_name] * n_splits_comparative_analysis
    display(pd.DataFrame(scores).select_dtypes(include=[float, int]).agg(['mean', 'std']))
    results = pd.concat([results, pd.DataFrame(scores)], ignore_index=True)

running K-Nearest Neighbors...


,fit_time,score_time,test_accuracy,test_precision_macro,test_recall_macro,test_f1_macro
mean,602.478025,6.834965,0.050757,0.016525,0.017777,0.015417
std,1.393340,3.303124,0.002284,0.001023,0.000594,0.000729


running Suport Vector Machines...


Exception in thread ExecutorManagerThread:
Traceback (most recent call last):
  File "C:\Users\vitor\anaconda3\Lib\site-packages\psutil\_pswindows.py", line 688, in wrapper
    return fun(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\vitor\anaconda3\Lib\site-packages\psutil\_pswindows.py", line 872, in kill
    return cext.proc_kill(self.pid)
           ^^^^^^^^^^^^^^^^^^^^^^^^
PermissionError: [WinError 5] Acesso negado: '(originated from OpenProcess)'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\vitor\anaconda3\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "C:\Users\vitor\anaconda3\Lib\site-packages\joblib\externals\loky\process_executor.py", line 560, in run
    self.terminate_broken(bpe)
  File "C:\Users\vitor\anaconda3\Lib\site-packages\joblib\externals\loky\process_executor.py", line 749, in terminate_broken
    self.kill_workers(reason="broken ex

TerminatedWorkerError: A worker process managed by the executor was unexpectedly terminated. This could be caused by a segmentation fault while calling the function or by an excessive memory usage causing the Operating System to kill the worker.


## 3.1 Resultados gerais

In [ ]:
def highlight_best(s, props=''):
    if s.name[1] != 'std':
        if s.name[0].endswith('time'):
            return np.where(s == np.nanmin(s.values), props, '')
        return np.where(s == np.nanmax(s.values), props, '')

display
(
    results
    .groupby('model_name')
    .agg(['mean', 'std']).T
    .style
    .apply(highlight_best, props='color:white;background-color:gray;font-weight: bold;', axis=1)
    .set_table_styles([{'selector': 'td', 'props': 'text-align: center;'}])
)

## 3.2 Persistência do modelo

In [ ]:
model_name, model_object, model_parameters  = [foo for foo in models if foo[0] == "K-Nearest Neighbors"][0] 


model_grid_search = GridSearchCV(
        estimator=model_object,
        param_grid=model_parameters,
        scoring=scoring,
        n_jobs=-1,
        cv=cross_validate_grid_search
    )

approach = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model_grid_search)
])

approach.fit(X, y) 

print(f"Hiper parâmetros do modelo: {approach.steps[1][1].best_params_}")

In [ ]:
joblib.dump(approach, '../models/model.joblib') 